In [12]:
import pandas as pd
import ast
import sys
import os
import numpy as np
from semopy import Model, calc_stats


In [13]:
from server_config import DATA_PATH
sys.path.append(os.path.abspath("functions"))

### 1. Therapist data

In [14]:
df_th = pd.read_csv(DATA_PATH + "/df_th_nopr_experience_corrected_200624_LB.csv", sep=";",low_memory = "False")

In [15]:
keep_cols_th = ['therapist_id', 't0_method_t_CBT','t0_method_t_PDT','t0_method_t_ST','t0_method_t_AP', 't0_appliedmethod_t_VT', 't0_appliedmethod_t_TP',
             't0_appliedmethod_t_AP', 't0_appliedmethod_t_ST','t0_appliedmethod_t_Other','t0_location_t','t0_gender_t',
 't0_relationship_t','t0_sexualorientation_t','t0_children_t','t0_age_t', 't0_experience_t','t0_license_t', 't0_bfi_t_1',
 't0_bfi_t_2','t0_bfi_t_3','t0_bfi_t_4','t0_bfi_t_5','t0_bfi_t_6','t0_bfi_t_7','t0_bfi_t_8','t0_bfi_t_9','t0_bfi_t_10','t0_practice_t']

In [16]:
labels = ["VT", "TP", "AP", "ST", "Other"]  

col = df_th["t0_appliedmethod_t_A"]

# 1) Parse strings like "[True, False, ...]" -> list
col = col.map(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# 2) Unwrap nested lists like [[True, False, ...]] -> [True, False, ...]
col = col.map(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) == 1 and isinstance(x[0], (list, tuple)) else x)

# 3) Safety: replace missing/invalid with all-False (or raise if you prefer)
def ensure_five(x):
    if isinstance(x, (list, tuple)) and len(x) == 5:
        return list(x)
    if pd.isna(x):
        return [False]*5

col = col.map(ensure_five)

# 4) Build the indicator columns
method_df = (
    pd.DataFrame(col.tolist(), index=df_th.index, columns=labels)
      .astype(int)
      .add_prefix("t0_appliedmethod_t_")
)

In [17]:
df_th = df_th.join(method_df)


In [18]:
df_th = df_th[keep_cols_th]

In [19]:
df_th

,therapist_id,t0_method_t_CBT,t0_method_t_PDT,t0_method_t_ST,t0_method_t_AP,t0_appliedmethod_t_VT,t0_appliedmethod_t_TP,t0_appliedmethod_t_AP,t0_appliedmethod_t_ST,t0_appliedmethod_t_Other,...,t0_bfi_t_2,t0_bfi_t_3,t0_bfi_t_4,t0_bfi_t_5,t0_bfi_t_6,t0_bfi_t_7,t0_bfi_t_8,t0_bfi_t_9,t0_bfi_t_10,t0_practice_t
0,9,1,0,0,0,1,0,0,0,0,...,3.0,4.0,3.0,3.0,3.0,3.0,4.0,3.0,3.0,NaN
1,10,1,0,0,0,1,0,0,0,0,...,4.0,4.0,1.0,3.0,5.0,2.0,4.0,1.0,3.0,NaN
2,11,1,0,0,0,1,0,0,0,0,...,5.0,5.0,1.0,3.0,4.0,4.0,5.0,1.0,4.0,NaN
3,12,0,1,0,0,0,0,0,0,1,...,4.0,3.0,2.0,5.0,3.0,4.0,4.0,2.0,5.0,NaN
4,13,1,0,0,0,1,0,0,0,1,...,4.0,3.0,2.0,2.0,4.0,4.0,4.0,1.0,5.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288,1214,1,0,0,0,1,0,0,0,1,...,4.0,3.0,2.0,3.0,5.0,3.0,3.0,2.0,5.0,NaN
289,1220,1,0,0,0,1,0,0,0,0,...,4.0,5.0,2.0,2.0,3.0,4.0,5.0,3.0,3.0,NaN
290,1244,1,0,0,0,1,0,0,0,1,...,5.0,2.0,3.0,2.0,5.0,4.0,4.0,3.0,5.0,NaN
291,1250,1,0,0,0,1,0,0,0,0,...,4.0,4.0,2.0,2.0,4.0,4.0,5.0,1.0,4.0,NaN


In [20]:
# BFI-10 items (1–5), reverse-code, then build subscale SUMS using df_th and t0-prefix
bfi_items = [f"t0_bfi_t_{i}" for i in range(1, 11)]
df_th[bfi_items] = df_th[bfi_items].apply(pd.to_numeric, errors="coerce")


# Subscale sums (2 items each)
df_th["t0_bfi_t_extraversion"]     = df_th[["t0_bfi_t_1", "t0_bfi_t_6"]].sum(axis=1)
df_th["t0_bfi_t_agreeableness"]    = df_th[["t0_bfi_t_2",   "t0_bfi_t_7"]].sum(axis=1)
df_th["t0_bfi_t_conscientiousness"]= df_th[["t0_bfi_t_3", "t0_bfi_t_8"]].sum(axis=1)
df_th["t0_bfi_t_neuroticism"]      = df_th[["t0_bfi_t_4", "t0_bfi_t_9"]].sum(axis=1)
df_th["t0_bfi_t_openness"]         = df_th[["t0_bfi_t_5", "t0_bfi_t_10"]].sum(axis=1)




### 2. Patient data

In [21]:
df_pat = pd.read_json(DATA_PATH + "/TONI_pat_corrected_20241025_raw.json")

In [22]:
keep_cols = (
    # IDs / study
    ['patient_id', 'therapist_id', 'u_registrationDate_p'] +

    # Demographics / baseline
    ['t0_age_p','t0_gender_p','t0_school_p','t0_employment_p','t0_location_p'] +
    [f"t0_therapy_p_{i}" for i in range(1, 4)] +
    ['t0_familyhistory_p','t0_meds_p','t0_relationship_p','t0_ses_p'] +
    [f"t0_migration_p_{i}" for i in range(1, 4)] +
    [f"surveys.t0.answers.1585.discrimination_{i}" for i in range(1, 6)] +
    [f"t0_disease_p_{c}" for c in list('ABCDEFG')] +

    # t0 scales
    [f"t0_phq_p_{i}" for i in range(1, 9)] +
    [f"t0_gad_p_{i}" for i in range(1, 7+1)] +
    [f"t0_swls_p_{i}" for i in range(1, 5+1)] +
    [f"t0_aqol_p_{i}" for i in range(1, 35+1)] +
    [f"t0_ede_p_{i}" for i in range(1, 8+1)] +
    (['t0_audit_p_filter'] + [f"t0_audit_p_{i}" for i in range(1, 3+1)]) +
    (['t0_dudit_p_filter'] + [f"t0_dudit_p_{i}" for i in range(1, 4+1)]) +
    [f"t0_msfq_p_{i}" for i in range(1, 5+1)] +
    [f"t0_pid_p_{i}" for i in range(1, 36+1)] +
    [f"t0_opd_p_{i}" for i in range(1, 12+1)] +
    [f"t0_ace_p_{i}" for i in range(1, 10+1)] +
    [f"t0_ask_p_{i}" for i in range(1, 3+1)] +
    [f"t0_mhse_p_{i}" for i in range(1, 6+1)] +

    # t1 scales
    [f"t1_phq_p_{i}" for i in range(1, 8+1)] +
    [f"t1_gad_p_{i}" for i in range(1, 7+1)] +
    [f"t1_multi_p_{i}" for i in range(1, 7+1)] +
    [f"t1_cpps_p_{i}" for i in range(1, 20+1)] +

    # t2 scales
    [f"t2_phq_p_{i}" for i in range(1, 8+1)] +
    [f"t2_gad_p_{i}" for i in range(1, 7+1)] +

    
    # t3 scales
    [f"t3_phq_p_{i}" for i in range(1, 8+1)] +
    [f"t3_gad_p_{i}" for i in range(1, 7+1)] +


    # alliance / matching

    [ 't0_change_p_1', 't0_change_p_2', 't0_dropout_p']
)


In [23]:
df_pat= df_pat[keep_cols]

#### 2.1 Calculate Subscales for self-report data

#### Outcome (PHQ + GAD)

In [24]:
# define item columns
phq_cols_t1 = [f"t1_phq_p_{i}" for i in range(1, 9)]
gad_cols_t1 = [f"t1_gad_p_{i}" for i in range(1, 8)]

mask_complete = df_pat[phq_cols_t1 + gad_cols_t1].notna().all(axis=1)
df_pat_clean = df_pat.loc[mask_complete].copy()

In [25]:
df_pat_clean["t1_phq8_total"] = df_pat_clean[phq_cols_t1].sum(axis=1)
df_pat_clean["t1_gad7_total"] = df_pat_clean[gad_cols_t1].sum(axis=1)

In [26]:
df_pat_clean["t1_distress_sum"] = (
    df_pat_clean["t1_phq8_total"] + df_pat_clean["t1_gad7_total"]
)

In [27]:
# define item columns
phq_cols_t0 = [f"t0_phq_p_{i}" for i in range(1, 9)]
gad_cols_t0 = [f"t0_gad_p_{i}" for i in range(1, 8)]

In [28]:
df_pat_clean["t0_phq8_p_sum"] = df_pat_clean[phq_cols_t0].sum(axis=1)
df_pat_clean["t0_gad7_p_sum"] = df_pat_clean[gad_cols_t0].sum(axis=1)

In [29]:
df_pat_clean["t0_distress_sum"] = (
    df_pat_clean["t0_phq8_p_sum"] + df_pat_clean["t0_gad7_p_sum"]
)

In [30]:
df_pat_clean["change_distress"] =  (df_pat_clean["t0_distress_sum"] - df_pat_clean["t1_distress_sum"])/df_pat_clean["t0_distress_sum"]*100

In [31]:
df_pat_clean = df_pat_clean.loc[df_pat_clean.change_distress.notna()]

#### SWLS Baseline

In [32]:
swls_cols_t0 = [f"t0_swls_p_{i}" for i in range(1, 6)]

In [33]:
df_pat_clean["t0_swls_p_sum"] = df_pat_clean[swls_cols_t0].sum(axis=1)

#### Aqol Baseline

In [34]:
aqol_map = {
    "t0_aqol_p_coping": ['t0_aqol_p_1', 't0_aqol_p_21', 't0_aqol_p_29'],
    "t0_aqol_p_relationship": ['t0_aqol_p_2', 't0_aqol_p_4', 't0_aqol_p_9', 't0_aqol_p_10',
                               't0_aqol_p_23', 't0_aqol_p_31', 't0_aqol_p_34'],
    "t0_aqol_p_independent_living": ['t0_aqol_p_3', 't0_aqol_p_15', 't0_aqol_p_19', 't0_aqol_p_30'],
    "t0_aqol_p_mental_health": ['t0_aqol_p_5', 't0_aqol_p_8', 't0_aqol_p_12', 't0_aqol_p_14',
                                't0_aqol_p_16', 't0_aqol_p_18', 't0_aqol_p_33', 't0_aqol_p_35'],
    "t0_aqol_p_pain": ['t0_aqol_p_6', 't0_aqol_p_22', 't0_aqol_p_24'],
    "t0_aqol_p_self_worth": ['t0_aqol_p_7', 't0_aqol_p_13', 't0_aqol_p_26'],
    "t0_aqol_p_happiness": ['t0_aqol_p_17', 't0_aqol_p_20', 't0_aqol_p_25', 't0_aqol_p_27'],
    "t0_aqol_p_senses": ['t0_aqol_p_11', 't0_aqol_p_28', 't0_aqol_p_32'],
}

In [35]:
df_pat_clean = df_pat_clean.assign(**{scale: df_pat_clean[items].mean(axis=1) for scale, items in aqol_map.items()})


#### EDE Baseline

In [36]:
df_pat_clean["t0_ede_p_sum"] = df_pat_clean[[f"t0_ede_p_{i}" for i in range(1, 9)]].sum(axis=1)


#### Audit Baseline

In [37]:
aud_items = [f"t0_audit_p_{i}" for i in range(1, 5)]
fcol = "t0_audit_p_filter"

# set all AUDIT item columns to 0 if filter == 0
df_pat_clean.loc[df_pat_clean[fcol].eq(0), aud_items] = 0

# sum AUDIT items only if filter == 1; otherwise set total to 0
df_pat_clean["t0_audit_p_sum"] = np.where(
    df_pat_clean[fcol].eq(1),
    df_pat_clean[aud_items].sum(axis=1),
    0
)

#### Dudit Baseline

In [38]:
dud_items = [f"t0_dudit_p_{i}" for i in range(1, 5)]
fcol = "t0_dudit_p_filter"

# set all AUDIT item columns to 0 if filter == 0
df_pat_clean.loc[df_pat_clean[fcol].eq(0), dud_items] = 0

# sum AUDIT items only if filter == 1; otherwise set total to 0
df_pat_clean["t0_dudit_p_sum"] = np.where(
    df_pat_clean[fcol].eq(1),
    df_pat_clean[dud_items].sum(axis=1),
    0
)

#### MFSQ Baseline

In [39]:
msfq_cols = [f"t0_msfq_p_{i}" for i in range(1, 6)]
df_pat_clean[msfq_cols] = df_pat_clean[msfq_cols].apply(pd.to_numeric, errors="coerce")

df_pat_clean["t0_msfq_p_sum"] = df_pat_clean[msfq_cols].sum(axis=1)

#### PID5BF+ Baseline

In [40]:
import pandas as pd

# -----------------------------
# PID-5 BF+ facets (2 items each) and domains (6 items each)
# -----------------------------
facet_map = {
    # Negative Affectivity facets
    "t0_pid_f_emo_lability":     ["t0_pid_p_1",  "t0_pid_p_19"],
    "t0_pid_f_anxiousness":      ["t0_pid_p_7",  "t0_pid_p_25"],
    "t0_pid_f_sep_insecurity":   ["t0_pid_p_13", "t0_pid_p_31"],  # note: 31 (not 30)

    # Detachment facets
    "t0_pid_f_withdrawal":       ["t0_pid_p_4",  "t0_pid_p_22"],
    "t0_pid_f_anhedonia":        ["t0_pid_p_10", "t0_pid_p_28"],
    "t0_pid_f_intimacy_avoid":   ["t0_pid_p_16", "t0_pid_p_34"],

    # Antagonism facets
    "t0_pid_f_manipulativeness": ["t0_pid_p_2",  "t0_pid_p_20"],
    "t0_pid_f_deceitfulness":    ["t0_pid_p_8",  "t0_pid_p_26"],
    "t0_pid_f_grandiosity":      ["t0_pid_p_14", "t0_pid_p_32"],

    # Disinhibition facets
    "t0_pid_f_irresponsibility": ["t0_pid_p_3",  "t0_pid_p_21"],
    "t0_pid_f_impulsivity":      ["t0_pid_p_9",  "t0_pid_p_27"],
    "t0_pid_f_distractibility":  ["t0_pid_p_15", "t0_pid_p_33"],

    # Psychoticism facets
    "t0_pid_f_unusual_beliefs":  ["t0_pid_p_5",  "t0_pid_p_23"],
    "t0_pid_f_eccentricity":     ["t0_pid_p_11", "t0_pid_p_29"],
    "t0_pid_f_perc_dysreg":      ["t0_pid_p_17", "t0_pid_p_35"],

    # Anankastia facets
    "t0_pid_f_perfectionism":    ["t0_pid_p_6",  "t0_pid_p_18"],
    "t0_pid_f_rigidity":         ["t0_pid_p_12", "t0_pid_p_24"],
    "t0_pid_f_orderliness":      ["t0_pid_p_30", "t0_pid_p_36"],
}

pid_map = {
    "t0_pid_p_negative_affect": ['t0_pid_p_1','t0_pid_p_7','t0_pid_p_13','t0_pid_p_19','t0_pid_p_25','t0_pid_p_31'],
    "t0_pid_p_antagonism":      ['t0_pid_p_2','t0_pid_p_8','t0_pid_p_14','t0_pid_p_20','t0_pid_p_26','t0_pid_p_32'],
    "t0_pid_p_disinhibition":   ['t0_pid_p_3','t0_pid_p_9','t0_pid_p_15','t0_pid_p_21','t0_pid_p_27','t0_pid_p_33'],
    "t0_pid_p_detachment":      ['t0_pid_p_4','t0_pid_p_10','t0_pid_p_16','t0_pid_p_22','t0_pid_p_28','t0_pid_p_34'],
    "t0_pid_p_psychoticism":    ['t0_pid_p_5','t0_pid_p_11','t0_pid_p_17','t0_pid_p_23','t0_pid_p_29','t0_pid_p_35'],
    "t0_pid_p_anankastia":      ['t0_pid_p_6','t0_pid_p_12','t0_pid_p_18','t0_pid_p_24','t0_pid_p_30','t0_pid_p_36'],
}

# -----------------------------
# 1) Coerce all PID item columns to numeric
# -----------------------------
item_cols = sorted({c for cols in (list(pid_map.values()) + list(facet_map.values())) for c in cols})
df_pat_clean[item_cols] = df_pat_clean[item_cols].apply(pd.to_numeric, errors="coerce")

# -----------------------------
# 2) Facet means (2 items each)
# -----------------------------
for facet, items in facet_map.items():
    df_pat_clean[facet] = df_pat_clean[items].mean(axis=1)

# -----------------------------
# 3) Domain means (6 items each) – your original approach
# -----------------------------
for scale, items in pid_map.items():
    df_pat_clean[scale] = df_pat_clean[items].mean(axis=1)

# Optional: domains from facet means (mathematically equivalent if no missingness)
domain_from_facets = {
    "t0_pid_p_negative_affect_from_facets": ["t0_pid_f_emo_lability", "t0_pid_f_anxiousness", "t0_pid_f_sep_insecurity"],
    "t0_pid_p_detachment_from_facets":      ["t0_pid_f_withdrawal", "t0_pid_f_anhedonia", "t0_pid_f_intimacy_avoid"],
    "t0_pid_p_antagonism_from_facets":      ["t0_pid_f_manipulativeness", "t0_pid_f_deceitfulness", "t0_pid_f_grandiosity"],
    "t0_pid_p_disinhibition_from_facets":   ["t0_pid_f_irresponsibility", "t0_pid_f_impulsivity", "t0_pid_f_distractibility"],
    "t0_pid_p_psychoticism_from_facets":    ["t0_pid_f_unusual_beliefs", "t0_pid_f_eccentricity", "t0_pid_f_perc_dysreg"],
    "t0_pid_p_anankastia_from_facets":      ["t0_pid_f_perfectionism", "t0_pid_f_rigidity", "t0_pid_f_orderliness"],
}
for scale, facets in domain_from_facets.items():
    df_pat_clean[scale] = df_pat_clean[facets].mean(axis=1)


/tmp/ipykernel_1461561/1290613486.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_pat_clean[scale] = df_pat_clean[facets].mean(axis=1)
/tmp/ipykernel_1461561/1290613486.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_pat_clean[scale] = df_pat_clean[facets].mean(axis=1)


#### OPD Baseline

In [41]:
# OPD-SFK domains → item columns
opd_map = {
    "t0_opd_p_self":         ['t0_opd_p_1', 't0_opd_p_2', 't0_opd_p_5', 't0_opd_p_8'],
    "t0_opd_p_contact":      ['t0_opd_p_4', 't0_opd_p_6', 't0_opd_p_10', 't0_opd_p_11'],
    "t0_opd_p_relationship": ['t0_opd_p_3', 't0_opd_p_7', 't0_opd_p_9', 't0_opd_p_12'],
}

# coerce item columns to numeric (e.g., "2" → 2; invalid → NaN)
opd_items = sorted({c for cols in opd_map.values() for c in cols})
df_pat_clean[opd_items] = df_pat_clean[opd_items].apply(pd.to_numeric, errors="coerce")

# domain sums (NaNs treated as 0 by default)
for scale, items in opd_map.items():
    df_pat_clean[scale] = df_pat_clean[items].mean(axis=1)

/tmp/ipykernel_1461561/4147894947.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_pat_clean[scale] = df_pat_clean[items].mean(axis=1)
/tmp/ipykernel_1461561/4147894947.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_pat_clean[scale] = df_pat_clean[items].mean(axis=1)
/tmp/ipykernel_1461561/4147894947.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

In [42]:
import pandas as pd
from semopy import Model, calc_stats

# ---------- (keep your preprocessing exactly as you have it) ----------
df_pat_clean = df_pat_clean.copy()

phq_cols_t0 = [f"t0_phq_p_{i}" for i in range(1, 9)]
gad_cols_t0 = [f"t0_gad_p_{i}" for i in range(1, 8)]

df_pat_clean[phq_cols_t0 + gad_cols_t0] = df_pat_clean[phq_cols_t0 + gad_cols_t0].apply(
    pd.to_numeric, errors="coerce"
)

df_pat_clean = df_pat_clean.assign(
    t0_phq_mean=df_pat_clean[phq_cols_t0].mean(axis=1),
    t0_gad_mean=df_pat_clean[gad_cols_t0].mean(axis=1),
)

pid_facets = [
    "t0_pid_f_emo_lability", "t0_pid_f_anxiousness", "t0_pid_f_sep_insecurity",
    "t0_pid_f_withdrawal", "t0_pid_f_anhedonia", "t0_pid_f_intimacy_avoid",
    "t0_pid_f_manipulativeness", "t0_pid_f_deceitfulness", "t0_pid_f_grandiosity",
    "t0_pid_f_irresponsibility", "t0_pid_f_impulsivity", "t0_pid_f_distractibility",
    "t0_pid_f_unusual_beliefs", "t0_pid_f_eccentricity", "t0_pid_f_perc_dysreg",
]
opd_domains = ["t0_opd_p_self", "t0_opd_p_contact", "t0_opd_p_relationship"]

required = pid_facets + opd_domains + ["t0_phq_mean", "t0_gad_mean"]
missing = [c for c in required if c not in df_pat_clean.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def meas(latent, indicators):
    return f"{latent} =~ " + " + ".join(indicators)

general_inds = pid_facets + ["t0_phq_mean", "t0_gad_mean"] + opd_domains

model_opd_phqgad_in_na = "\n".join([
    meas("P_Func", general_inds),

    meas("PID_Na", ["t0_pid_f_emo_lability", "t0_pid_f_anxiousness", "t0_pid_f_sep_insecurity"]),
    meas("PID_Dt", ["t0_pid_f_withdrawal", "t0_pid_f_anhedonia", "t0_pid_f_intimacy_avoid"]),
    meas("PID_An", ["t0_pid_f_manipulativeness", "t0_pid_f_deceitfulness", "t0_pid_f_grandiosity"]),
    meas("PID_Di", ["t0_pid_f_irresponsibility", "t0_pid_f_impulsivity", "t0_pid_f_distractibility"]),
    meas("PID_Ps", ["t0_pid_f_unusual_beliefs", "t0_pid_f_eccentricity", "t0_pid_f_perc_dysreg"]),

    "P_Func ~~ 0*PID_Na",
    "P_Func ~~ 0*PID_Dt",
    "P_Func ~~ 0*PID_An",
    "P_Func ~~ 0*PID_Di",
    "P_Func ~~ 0*PID_Ps",
])

# ---------- REFIT WITH MLW (NOT FIML) ----------
mod = Model(model_opd_phqgad_in_na)
mod.fit(df_pat_clean, obj="MLW")   # key change

fit_stats = calc_stats(mod)
print(fit_stats.T)

# Conventional SEs
est_ml = mod.inspect(std_est=True)

# Robust (sandwich) SEs (MLR-esque)
est_rb = mod.inspect(std_est=True, se_robust=True)  # supported in semopy docs :contentReference[oaicite:2]{index=2}

print(est_rb.head(60))


                     Value
DoF             145.000000
DoF Baseline    190.000000
chi2           1190.072827
chi2 p-value      0.000000
chi2 Baseline  6074.663364
CFI               0.822407
GFI               0.804092
AGFI              0.743293
NFI               0.804092
TLI               0.767292
RMSEA             0.086467
AIC             127.533528
BIC             444.221854
LogLik            1.233236
                         lval  op                   rval  Estimate  Est. Std  \
0       t0_pid_f_emo_lability   ~                 P_Func  1.000000  0.403324   
1       t0_pid_f_emo_lability   ~                 PID_Na  1.000000  0.310139   
2        t0_pid_f_anxiousness   ~                 P_Func  1.423995  0.595555   
3        t0_pid_f_anxiousness   ~                 PID_Na  0.996320  0.320416   
4     t0_pid_f_sep_insecurity   ~                 P_Func  1.182588  0.439348   
5     t0_pid_f_sep_insecurity   ~                 PID_Na  1.050806  0.300193   
6         t0_pid_f_withdrawal   ~  

#### ACE Baseline

In [43]:
ace_cols = [f"t0_ace_p_{i}" for i in range(1, 11)]

# ensure numeric (e.g., "1" -> 1; invalid -> NaN)
df_pat_clean[ace_cols] = df_pat_clean[ace_cols].apply(pd.to_numeric, errors="coerce")

df_pat_clean["t0_ace_p_sum"] = df_pat_clean[ace_cols].sum(axis=1)

#### ASK Baseline

In [44]:
ask_cols = [f"t0_ask_p_{i}" for i in range(1, 4)]
df_pat_clean[ask_cols] = df_pat_clean[ask_cols].apply(pd.to_numeric, errors="coerce")

df_pat_clean["t0_ask_p_sum"] = df_pat_clean[ask_cols].sum(axis=1)

#### MHSE Baseline

In [45]:
mhse_cols = [f"t0_mhse_p_{i}" for i in range(1, 7)]
df_pat_clean[mhse_cols] = df_pat_clean[mhse_cols].apply(pd.to_numeric, errors="coerce")

df_pat_clean["t0_mhse_p_sum"] = df_pat_clean[mhse_cols].sum(axis=1)

#### Multi T1

In [46]:
# define the 7 MULTI item columns once
multi_cols = [f"t1_multi_p_{i}" for i in range(1, 8)]

# 1) ensure the same columns are numeric
df_pat_clean[multi_cols] = df_pat_clean[multi_cols].apply(pd.to_numeric, errors="coerce")

# 2) compute therapist-level means and attach as 7 new columns to every row
multi_avg = (
    df_pat_clean
    .groupby("therapist_id")[multi_cols]
    .transform("mean")
)
multi_avg.columns = [c.replace("t1_multi_p_", "multi_avg_") for c in multi_avg.columns]

# 3) append to your dataframe (now you have multi_avg_1 .. multi_avg_7)
df_pat_clean = pd.concat([df_pat_clean, multi_avg], axis=1)
df_pat_clean["multi_sum"] = df_pat_clean[["multi_avg_1", "multi_avg_2", "multi_avg_3", "multi_avg_4", "multi_avg_5","multi_avg_6","multi_avg_7"]].sum(axis=1)

#### CPPS

In [47]:
cpps_cols = [f"t1_cpps_p_{i}" for i in range(1, 21)]

df_pat_clean[cpps_cols] = df_pat_clean[cpps_cols].apply(pd.to_numeric, errors="coerce")
cpps_avg = (
    df_pat_clean
    .groupby("therapist_id")[cpps_cols]
    .transform("mean"))
cpps_avg.columns = [c.replace("t1_cpps_p_", "cpps_avg_") for c in cpps_avg.columns]
df_pat_clean = pd.concat([df_pat_clean, cpps_avg], axis=1)

df_pat_clean["tx_cpps_p_pd"] = df_pat_clean[['cpps_avg_1', 'cpps_avg_4', 'cpps_avg_5', 'cpps_avg_7',
'cpps_avg_8', 'cpps_avg_10', 'cpps_avg_13', 'cpps_avg_14', 'cpps_avg_16', 'cpps_avg_19']].sum(axis=1)

df_pat_clean["tx_cpps_p_vt"] = df_pat_clean[['cpps_avg_2', 'cpps_avg_3', 'cpps_avg_6', 'cpps_avg_9',
'cpps_avg_11', 'cpps_avg_12', 'cpps_avg_15', 'cpps_avg_17', 'cpps_avg_18', 'cpps_avg_20']].sum(axis=1)

### Intake 

### Concatenate patient and therapist df

In [48]:
df_all = pd.merge(df_pat_clean, df_th, on="therapist_id", how= "left", validate = "many_to_one")

In [49]:
df_all["u_registrationDate_p_datetime"] = pd.to_datetime(
    df_all["u_registrationDate_p"], errors="coerce", dayfirst=True, utc=True
)
iso = df_all["u_registrationDate_p_datetime"].dt.isocalendar()
df_all["intake_week"]  = iso.week.astype("Int64")
df_all["intake_month"] = df_all["u_registrationDate_p_datetime"].dt.month.astype("Int64")
df_all["intake_year"]  = df_all["u_registrationDate_p_datetime"].dt.year.astype("Int64").astype("string")

In [50]:
patient_numeric=['t0_age_p', 't0_ses_p', 'surveys.t0.answers.1585.discrimination_1', 'surveys.t0.answers.1585.discrimination_2',
 'surveys.t0.answers.1585.discrimination_3', 'surveys.t0.answers.1585.discrimination_4', 'surveys.t0.answers.1585.discrimination_5',
  't0_phq_p_1', 't0_phq_p_2', 't0_phq_p_3', 't0_phq_p_4', 't0_phq_p_5', 't0_phq_p_6', 't0_phq_p_7', 't0_phq_p_8',
 't0_gad_p_1', 't0_gad_p_2', 't0_gad_p_3', 't0_gad_p_4', 't0_gad_p_5', 't0_gad_p_6', 't0_gad_p_7','t0_swls_p_1',
 't0_swls_p_2', 't0_swls_p_3', 't0_swls_p_4', 't0_swls_p_5', 't0_aqol_p_1','t0_aqol_p_2', 't0_aqol_p_3',
 't0_aqol_p_4', 't0_aqol_p_5', 't0_aqol_p_6', 't0_aqol_p_7', 't0_aqol_p_8', 't0_aqol_p_9', 't0_aqol_p_10',
 't0_aqol_p_11', 't0_aqol_p_12', 't0_aqol_p_13', 't0_aqol_p_14', 't0_aqol_p_15', 't0_aqol_p_16', 't0_aqol_p_17',
 't0_aqol_p_18', 't0_aqol_p_19', 't0_aqol_p_20', 't0_aqol_p_21', 't0_aqol_p_22', 't0_aqol_p_23', 't0_aqol_p_24',
 't0_aqol_p_25', 't0_aqol_p_26', 't0_aqol_p_27', 't0_aqol_p_28', 't0_aqol_p_29', 't0_aqol_p_30', 't0_aqol_p_31',
 't0_aqol_p_32', 't0_aqol_p_33', 't0_aqol_p_34', 't0_aqol_p_35', 't0_audit_p_1', 't0_audit_p_2', 't0_audit_p_3',
  't0_dudit_p_1', 't0_dudit_p_2', 't0_dudit_p_3', 't0_dudit_p_4', 't0_msfq_p_1', 't0_msfq_p_2', 't0_msfq_p_3',
 't0_msfq_p_4', 't0_msfq_p_5', 't0_pid_p_1', 't0_pid_p_2', 't0_pid_p_3', 't0_pid_p_4', 't0_pid_p_5', 't0_pid_p_6',
 't0_pid_p_7', 't0_pid_p_8', 't0_pid_p_9', 't0_pid_p_10', 't0_pid_p_11', 't0_pid_p_12', 't0_pid_p_13', 't0_pid_p_14',
 't0_pid_p_15', 't0_pid_p_16', 't0_pid_p_17', 't0_pid_p_18', 't0_pid_p_19', 't0_pid_p_20', 't0_pid_p_21',
 't0_pid_p_22', 't0_pid_p_23', 't0_pid_p_24', 't0_pid_p_25', 't0_pid_p_26', 't0_pid_p_27', 't0_pid_p_28',
 't0_pid_p_29', 't0_pid_p_30', 't0_pid_p_31', 't0_pid_p_32', 't0_pid_p_33', 't0_pid_p_34', 't0_pid_p_35',
 't0_pid_p_36', 't0_opd_p_1', 't0_opd_p_2', 't0_opd_p_3', 't0_opd_p_4', 't0_opd_p_5', 't0_opd_p_6', 't0_opd_p_7',
 't0_opd_p_8', 't0_opd_p_9', 't0_opd_p_10', 't0_opd_p_11', 't0_opd_p_12', 't0_ace_p_1', 't0_ace_p_2',
 't0_ace_p_3', 't0_ace_p_4', 't0_ace_p_5', 't0_ace_p_6', 't0_ace_p_7', 't0_ace_p_8', 't0_ace_p_9', 't0_ace_p_10',
 't0_ask_p_1', 't0_ask_p_2', 't0_ask_p_3', 't0_mhse_p_1', 't0_mhse_p_2', 't0_mhse_p_3', 't0_mhse_p_4', 't0_mhse_p_5',
 't0_mhse_p_6', 't0_change_p_1', 't0_change_p_2', 't0_dropout_p', 't0_phq8_p_sum', 't0_gad7_p_sum', 't0_swls_p_sum',
 't0_aqol_p_coping','t0_aqol_p_relationship', 't0_aqol_p_independent_living', 't0_aqol_p_mental_health',
 't0_aqol_p_pain', 't0_aqol_p_self_worth', 't0_aqol_p_happiness', 't0_aqol_p_senses', 't0_audit_p_sum',
 't0_dudit_p_sum', 't0_msfq_p_sum', 't0_pid_p_negative_affect', 't0_pid_p_antagonism', 't0_pid_p_disinhibition',
 't0_pid_p_detachment', 't0_pid_p_psychoticism', 't0_pid_p_anankastia', 't0_opd_p_self', 't0_opd_p_contact',
 't0_opd_p_relationship', 't0_ace_p_sum', 't0_ask_p_sum', 't0_mhse_p_sum']

patient_categorical=['t0_gender_p', 't0_school_p', 't0_employment_p', 't0_location_p', 't0_therapy_p_1', 't0_therapy_p_2',
 't0_therapy_p_3', 't0_familyhistory_p', 't0_meds_p', 't0_relationship_p', 't0_migration_p_1',
 't0_migration_p_2', 't0_migration_p_3', 't0_disease_p_A', 't0_disease_p_B', 't0_disease_p_C',
 't0_disease_p_D', 't0_disease_p_E', 't0_disease_p_F', 't0_disease_p_G']

In [51]:
miss_summary = (df_all[patient_numeric + patient_categorical]
                .isna()
                .sum()
                .rename("n_nan")
                .to_frame())
miss_summary["pct_nan"] = miss_summary["n_nan"] / len(df_all) * 100
miss_summary.sort_values("pct_nan", ascending=False).head(50)


,n_nan,pct_nan
t0_therapy_p_3,459,47.564767
t0_pid_p_23,309,32.020725
t0_pid_p_25,309,32.020725
t0_pid_p_26,309,32.020725
t0_pid_p_27,309,32.020725
t0_pid_p_24,309,32.020725
t0_pid_p_33,309,32.020725
t0_pid_p_30,309,32.020725
t0_pid_p_31,309,32.020725
t0_pid_p_32,309,32.020725


#### Remove therapists with missing side features

In [52]:
df_all = df_all.loc[df_all.t0_experience_t.notna()]

In [53]:
mask_bad_age = df_all["t0_age_t"].notna() & (df_all["t0_age_t"] < 23)
n_bad = int(mask_bad_age.sum())
print(f"[repair] t0_age_t: setting {n_bad} implausible values (<23) to NaN")

df_all.loc[mask_bad_age, "t0_age_t"] = np.nan


[repair] t0_age_t: setting 92 implausible values (<23) to NaN


In [54]:
df_all["n_patients"] = df_all.groupby("therapist_id")["patient_id"].transform("nunique")

In [55]:
methods  = ["t0_method_t_CBT","t0_method_t_PDT","t0_method_t_ST","t0_method_t_AP"]
applied  = ["t0_appliedmethod_t_VT","t0_appliedmethod_t_TP","t0_appliedmethod_t_AP",
            "t0_appliedmethod_t_ST","t0_appliedmethod_t_Other"]

tmp = (df_all.assign(**{c: pd.to_numeric(df_all[c], errors="coerce").fillna(0).astype(int)
                        for c in methods + applied})
             .groupby("therapist_id")[methods + applied].max())

therapist_counts = tmp.assign(
    n_methods=tmp[methods].sum(1),
    n_appliedmethods=tmp[applied].sum(1)
)[["n_methods","n_appliedmethods"]].reset_index()
df_all = df_all.merge(therapist_counts, on="therapist_id", how="left", validate="many_to_one")

In [56]:
therapist_keep = ['n_patients','t0_age_t','t0_experience_t',
        't0_bfi_t_extraversion','t0_bfi_t_agreeableness','t0_bfi_t_conscientiousness',
        't0_bfi_t_neuroticism','t0_bfi_t_openness', 
        't0_method_t_CBT','t0_method_t_PDT','t0_method_t_ST','n_methods','n_appliedmethods','t0_method_t_AP',
        't0_appliedmethod_t_VT','t0_appliedmethod_t_TP','t0_appliedmethod_t_AP',
        't0_appliedmethod_t_ST','t0_appliedmethod_t_Other',
        't0_location_t','t0_gender_t','t0_relationship_t','t0_sexualorientation_t','t0_children_t','t0_practice_t']

In [57]:
 patient_keep= ['t0_age_p','t0_gender_p','t0_phq8_p_sum','t0_gad7_p_sum', 't0_school_p', 't0_employment_p', 't0_location_p',
                't0_relationship_p', 't0_migration_p_1','t0_ses_p',
 't0_migration_p_2', 't0_migration_p_3', 't0_disease_p_A', 't0_disease_p_B', 't0_disease_p_C',
 't0_disease_p_D', 't0_disease_p_E', 't0_disease_p_F', 't0_disease_p_G', 't0_pid_p_negative_affect', 't0_pid_p_antagonism', 't0_pid_p_disinhibition',
 't0_pid_p_detachment', 't0_pid_p_psychoticism', 't0_pid_p_anankastia']

In [58]:
df_all_patients = df_all[patient_keep].copy()

In [59]:
df_all_therapists = df_all[["therapist_id"] + therapist_keep].drop_duplicates(subset="therapist_id").reset_index(drop=True)

In [60]:
#df_all_therapists.to_csv("therapist_data.csv", index=False)

In [61]:
df_all_patients.to_csv("patient_data.csv", index=False)

In [62]:
b

NameError: name 'b' is not defined

#### Calculate therapist effects

In [ ]:
# therapist_icc_t1_distress.py
import numpy as np
import pandas as pd
import warnings
import statsmodels.formula.api as smf
import re
from typing import List, Optional

# ---- robust variance-component getter (handles statsmodels version quirks) ----
def _get_vcomp(fit, term="patient"):
    sigma2 = np.nan
    vcomp = getattr(fit, "vcomp", None)
    if vcomp is None:
        return sigma2
    names = getattr(fit, "vcomp_names", None) or getattr(fit.model, "vcomp_names", None)
    if names is not None and len(names) == len(vcomp):
        if term in names:
            return float(vcomp[names.index(term)])
        for i, nm in enumerate(names):
            s = str(nm)
            if term in s or s.endswith(term):
                return float(vcomp[i])
    else:
        try:
            if hasattr(vcomp, "__len__") and len(vcomp) == 1:
                return float(vcomp[0])
        except Exception:
            pass
    return sigma2

def _extract_parts(fit):
    # therapist variance
    try:
        sigma2_ther = float(getattr(fit, "cov_re", np.array([[np.nan]]))[0][0])
    except Exception:
        sigma2_ther = float(fit.cov_re.iloc[0, 0])
    # patient variance + residual
    sigma2_pat  = _get_vcomp(fit, "patient")
    sigma2_res  = float(fit.scale)
    denom = sigma2_ther + sigma2_pat + sigma2_res
    icc_ther = sigma2_ther / denom if denom > 0 else np.nan
    return icc_ther, sigma2_ther, sigma2_pat, sigma2_res

def fit_icc_t1_distress(df: pd.DataFrame,
                        outcome_col: str = "t1_distress_sum",
                        therapist_col: str = "therapist_id",
                        patient_col: str = "patient_id",
                        return_blups: bool = True):
    """Null model: y ~ 1 + (1|therapist) + (1|patient)."""
    d = df[[outcome_col, therapist_col, patient_col]].dropna().copy()
    d[therapist_col] = d[therapist_col].astype(str)
    d[patient_col]   = d[patient_col].astype(str)

    if d[therapist_col].nunique() < 2 or d[patient_col].nunique() < 2:
        return {"model":"null","outcome":outcome_col,"icc_therapist":np.nan,"n":len(d),
                "n_therapists":d[therapist_col].nunique(),"n_patients":d[patient_col].nunique(),
                "sigma2_therapist":np.nan,"sigma2_patient":np.nan,"sigma2_resid":np.nan,
                "converged":False,"method":None,"blups":None,"message":"Insufficient groups"}

    vc_formula = {"patient": f"0 + C({patient_col})"}
    methods = ["lbfgs","bfgs","cg","powell","nm"]
    fit = None; used_method=None; last_err=None
    for m in methods:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = smf.mixedlm(f"{outcome_col} ~ 1", d,
                                    groups=d[therapist_col], re_formula="1",
                                    vc_formula=vc_formula)
                fit = model.fit(method=m, reml=True, maxiter=2000, disp=0)
            if fit.converged:
                used_method = m
                break
        except Exception as e:
            last_err = str(e); fit=None

    if fit is None or not fit.converged:
        return {"model":"null","outcome":outcome_col,"icc_therapist":np.nan,"n":len(d),
                "n_therapists":d[therapist_col].nunique(),"n_patients":d[patient_col].nunique(),
                "sigma2_therapist":np.nan,"sigma2_patient":np.nan,"sigma2_resid":np.nan,
                "converged":False,"method":None,"blups":None,"message":last_err or "Did not converge"}

    icc_ther, v_t, v_p, v_e = _extract_parts(fit)

    blups_df = None
    if return_blups:
        re = fit.random_effects
        blups_df = pd.DataFrame({therapist_col: list(re.keys()),
                                 "blup_intercept": [float(v[0]) for v in re.values()]})

    return {"model":"null","outcome":outcome_col,"icc_therapist":float(np.round(icc_ther,3)),
            "n":int(len(d)),"n_therapists":int(d[therapist_col].nunique()),
            "n_patients":int(d[patient_col].nunique()),
            "sigma2_therapist":v_t,"sigma2_patient":v_p,"sigma2_resid":v_e,
            "converged":bool(fit.converged),"method":used_method,"blups":blups_df,"message":"ok"}

def fit_icc_t1_distress_adjusted(df: pd.DataFrame,
                                 outcome_col: str = "t1_distress_sum",
                                 baseline_col: str = "t0_distress_sum",
                                 therapist_col: str = "therapist_id",
                                 patient_col: str = "patient_id",
                                 return_blups: bool = False):
    """Baseline-adjusted: y_post ~ y_base + (1|therapist) + (1|patient)."""
    if baseline_col not in df.columns:
        return {"model":"baseline_adjusted","outcome":outcome_col,"baseline":baseline_col,
                "icc_therapist":np.nan,"message":f"Missing baseline column: {baseline_col}",
                "converged":False,"method":None,"blups":None}

    d = df[[outcome_col, baseline_col, therapist_col, patient_col]].dropna().copy()
    d[therapist_col] = d[therapist_col].astype(str)
    d[patient_col]   = d[patient_col].astype(str)

    if d[therapist_col].nunique() < 2 or d[patient_col].nunique() < 2:
        return {"model":"baseline_adjusted","outcome":outcome_col,"baseline":baseline_col,
                "icc_therapist":np.nan,"n":len(d),
                "n_therapists":d[therapist_col].nunique(),"n_patients":d[patient_col].nunique(),
                "sigma2_therapist":np.nan,"sigma2_patient":np.nan,"sigma2_resid":np.nan,
                "converged":False,"method":None,"blups":None,"message":"Insufficient groups"}

    vc_formula = {"patient": f"0 + C({patient_col})"}
    methods = ["lbfgs","bfgs","cg","powell","nm"]
    fit=None; used_method=None; last_err=None
    for m in methods:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = smf.mixedlm(f"{outcome_col} ~ {baseline_col}", d,
                                    groups=d[therapist_col], re_formula="1",
                                    vc_formula=vc_formula)
                fit = model.fit(method=m, reml=True, maxiter=2000, disp=0)
            if fit.converged:
                used_method = m
                break
        except Exception as e:
            last_err = str(e); fit=None

    if fit is None or not fit.converged:
        return {"model":"baseline_adjusted","outcome":outcome_col,"baseline":baseline_col,
                "icc_therapist":np.nan,"n":len(d),
                "n_therapists":d[therapist_col].nunique(),"n_patients":d[patient_col].nunique(),
                "sigma2_therapist":np.nan,"sigma2_patient":np.nan,"sigma2_resid":np.nan,
                "converged":False,"method":None,"blups":None,"message":last_err or "Did not converge"}

    icc_ther, v_t, v_p, v_e = _extract_parts(fit)

    blups_df = None
    if return_blups:
        re = fit.random_effects
        blups_df = pd.DataFrame({therapist_col: list(re.keys()),
                                 "blup_intercept": [float(v[0]) for v in re.values()]})

    return {"model":"baseline_adjusted","outcome":outcome_col,"baseline":baseline_col,
            "icc_therapist":float(np.round(icc_ther,3)),
            "n":int(len(d)),"n_therapists":int(d[therapist_col].nunique()),
            "n_patients":int(d[patient_col].nunique()),
            "sigma2_therapist":v_t,"sigma2_patient":v_p,"sigma2_resid":v_e,
            "converged":bool(fit.converged),"method":used_method,"blups":blups_df,"message":"ok"}

# ----------------- run (expects df_all in memory) -----------------
res_null = fit_icc_t1_distress(df_all, outcome_col="t1_distress_sum",
                               therapist_col="therapist_id", patient_col="patient_id",
                               return_blups=True)

res_adj  = fit_icc_t1_distress_adjusted(df_all,
                                        outcome_col="t1_distress_sum",
                                        baseline_col="t0_distress_sum",
                                        therapist_col="therapist_id",
                                        patient_col="patient_id",
                                        return_blups=False)

# Save results
pd.DataFrame([{k: v for k, v in res_null.items() if k != "blups"}]).to_csv(
    "ICC_therapist_t1_distress_null.csv", index=False
)
if isinstance(res_null.get("blups"), pd.DataFrame):
    res_null["blups"].to_csv("Therapist_BLUPs_t1_distress_null.csv", index=False)

pd.DataFrame([{k: v for k, v in res_adj.items() if k != "blups"}]).to_csv(
    "ICC_therapist_t1_distress_baseline_adjusted.csv", index=False
)

print("[NULL]    ICC_therapist =", res_null["icc_therapist"],
      "| n =", res_null["n"], "| therapists =", res_null["n_therapists"],
      "| patients =", res_null["n_patients"], "| converged =", res_null["converged"],
      "| method =", res_null["method"])

print("[ADJUST]  ICC_therapist =", res_adj["icc_therapist"],
      "| n =", res_adj.get("n"), "| therapists =", res_adj.get("n_therapists"),
      "| patients =", res_adj.get("n_patients"), "| converged =", res_adj["converged"],
      "| method =", res_adj["method"])


In [ ]:
# ---------------------------
# 1) Reliability estimators
# ---------------------------
def cronbach_alpha(df_items: pd.DataFrame) -> float:
    """Cronbach's alpha on item *covariance* (not correlations). Listwise deletion."""
    X = df_items.dropna(axis=0, how="any")
    k = X.shape[1]
    if k < 2:
        return np.nan
    cov = np.cov(X.values, rowvar=False, ddof=1)
    var_sum = np.trace(cov)
    total_var = cov.sum()  # variance of the sum score
    alpha = (k / (k - 1.0)) * (1.0 - (var_sum / total_var))
    return float(alpha)

def mcdonalds_omega_total(df_items: pd.DataFrame) -> float:
    """McDonald's ω_total via 1-factor ML (requires factor_analyzer). Falls back to NaN if not installed."""
    try:
        from factor_analyzer import FactorAnalyzer
    except Exception:
        return np.nan
    X = df_items.dropna(axis=0, how="any")
    if X.shape[1] < 2 or X.shape[0] < 50:
        return np.nan
    # Standardize items to unit variance for stable loadings
    Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
    fa = FactorAnalyzer(n_factors=1, method="ml", rotation=None)
    fa.fit(Z.values)
    loadings = fa.loadings_.reshape(-1)                   # λ_i
    uniques = fa.get_uniquenesses().reshape(-1)           # ψ_i
    # ω_total ≈ (Σλ_i)^2 / ((Σλ_i)^2 + Σψ_i)  (unidimensional approximation on standardized items)
    omega = (loadings.sum() ** 2) / ((loadings.sum() ** 2) + uniques.sum())
    # Guard against numerical drift
    return float(np.clip(omega, 0.0, 1.0))

# -----------------------------------------
# 2) Auto-detect the 16 T1 item columns
# -----------------------------------------
def guess_t1_item_cols(df: pd.DataFrame) -> List[str]:
    """
    Heuristics for PHQ-9 + GAD-7 item names at T1.
    Adjust patterns if your columns differ.
    """
    pats = [
        r"^t1_.*phq.*_([1-9])$",                # e.g., t1_phq_p_1
        r"^t1_.*gad.*_([1-7])$",                # e.g., t1_gad_p_1
    ]
    candidates = []
    for c in df.columns:
        cname = str(c).lower()
        if any(re.search(p, cname) for p in pats):
            # keep only numeric dtypes
            if pd.api.types.is_numeric_dtype(df[c]):
                candidates.append(c)
    # try to keep exactly one PHQ set (9) and one GAD set (7)
    phq = [c for c in candidates if "phq" in c.lower()]
    gad = [c for c in candidates if "gad" in c.lower()]
    # sort by trailing number if present
    def sort_by_trailing_num(cols):
        def keyf(x):
            m = re.search(r"(\d+)\s*$", str(x))
            return int(m.group(1)) if m else 999
        return sorted(cols, key=keyf)
    phq = sort_by_trailing_num(phq)[:9]
    gad = sort_by_trailing_num(gad)[:7]
    items = phq + gad
    return items

# -----------------------------------------------------------
# 3) Compute SEM & MDC95 for your composite (t1_distress_sum)
# -----------------------------------------------------------
def sem_mdc_from_items_and_sum(
    df: pd.DataFrame,
    sum_col: str = "t1_distress_sum",
    item_cols: Optional[List[str]] = None,
    prefer_omega: bool = True
):
    if item_cols is None:
        item_cols = guess_t1_item_cols(df)
    if len(item_cols) != 15:
        raise ValueError(f"Expected 16 T1 items, found {len(item_cols)}: {item_cols}")

    # Reliability
    alpha = cronbach_alpha(df[item_cols])
    omega = mcdonalds_omega_total(df[item_cols])
    if prefer_omega and np.isfinite(omega):
        reliability = omega
        rel_name = "omega_total"
    else:
        reliability = alpha
        rel_name = "cronbach_alpha"

    # SD of composite in your sample
    y = pd.to_numeric(df[sum_col], errors="coerce").dropna()
    sd = float(y.std(ddof=1))

    # SEM & MDC95
    sem = sd * np.sqrt(max(0.0, 1.0 - reliability))
    mdc95 = 1.96 * np.sqrt(2.0) * sem

    return {
        "sum_col": sum_col,
        "items_used": item_cols,
        "reliability_name": rel_name,
        "reliability": float(reliability) if np.isfinite(reliability) else np.nan,
        "sd_sum": sd,
        "sem": float(sem),
        "mdc95": float(mdc95),
        "notes": "Reliability from items; SD from composite at T1 (listwise deletion for reliability)."}


In [ ]:
sem_info = sem_mdc_from_items_and_sum(df_all, sum_col="t1_distress_sum", item_cols=None, prefer_omega=True)
# print(sem_info)

In [ ]:
sem_info